# GraphSAGE Link Prediction — PyTorch Geometric → ONNX

Train a 2-layer GraphSAGE encoder for link prediction, export encoder to ONNX.

**Requires**: Kaggle GPU accelerator

**Input**: Graph TSV + embeddings NPZ (from notebooks 03)
**Output**: `model.onnx` + `predicted_edges.tsv`

In [ ]:
!pip install -q torch-geometric torch-scatter torch-sparse

In [ ]:
import os
import numpy as np

OUTPUT_DIR = "/kaggle/working/models/gnn_v1"
GRAPH_TSV = "/kaggle/input/cooccurrence-graph/graph.tsv"
EMBEDDINGS_NPZ = "/kaggle/input/embeddings/embeddings.npz"
HIDDEN_DIM = 128
EPOCHS = 100
LR = 0.01
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling

# Load embeddings as initial node features
data_npz = np.load(EMBEDDINGS_NPZ)
node_features = torch.tensor(data_npz["embeddings"], dtype=torch.float)
node_ids = list(data_npz["node_ids"])
id_to_idx = {nid: i for i, nid in enumerate(node_ids)}

# Load edges
src, dst = [], []
with open(GRAPH_TSV) as f:
    next(f)  # skip header
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2 and parts[0] in id_to_idx and parts[1] in id_to_idx:
            src.append(id_to_idx[parts[0]])
            dst.append(id_to_idx[parts[1]])
            src.append(id_to_idx[parts[1]])  # undirected
            dst.append(id_to_idx[parts[0]])

edge_index = torch.tensor([src, dst], dtype=torch.long)
data = Data(x=node_features, edge_index=edge_index)
print(f"Nodes: {data.num_nodes}, Edges: {data.edge_index.shape[1]}")

In [ ]:
# Train/val split — hold out 10% of edges
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(num_val=0.1, num_test=0.0, is_undirected=True, add_negative_train_samples=True)
train_data, val_data, _ = transform(data)
print(f"Train edges: {train_data.edge_label_index.shape[1]}, Val edges: {val_data.edge_label_index.shape[1]}")

In [ ]:
class GraphSAGEEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

encoder = GraphSAGEEncoder(node_features.shape[1], HIDDEN_DIM).to(device)
train_data = train_data.to(device)
val_data = val_data.to(device)

optimizer = torch.optim.Adam(encoder.parameters(), lr=LR)

In [ ]:
from sklearn.metrics import roc_auc_score

def train_epoch():
    encoder.train()
    optimizer.zero_grad()
    z = encoder(train_data.x, train_data.edge_index)
    
    # Dot product decoder
    edge_label_index = train_data.edge_label_index
    edge_label = train_data.edge_label.float()
    
    out = (z[edge_label_index[0]] * z[edge_label_index[1]]).sum(dim=-1)
    loss = F.binary_cross_entropy_with_logits(out, edge_label)
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def eval_auc(data_split):
    encoder.eval()
    z = encoder(data_split.x, data_split.edge_index)
    edge_label_index = data_split.edge_label_index
    out = (z[edge_label_index[0]] * z[edge_label_index[1]]).sum(dim=-1).sigmoid()
    return roc_auc_score(data_split.edge_label.cpu(), out.cpu())

for epoch in range(1, EPOCHS + 1):
    loss = train_epoch()
    if epoch % 10 == 0:
        val_auc = eval_auc(val_data)
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | Val AUC: {val_auc:.4f}")

In [ ]:
# Predict new edges (top-K non-connected pairs by score)
@torch.no_grad()
def predict_new_edges(k=10000):
    encoder.eval()
    z = encoder(data.to(device).x, data.to(device).edge_index).cpu()
    
    # Sample random node pairs not in the graph
    existing = set()
    ei = data.edge_index.t().tolist()
    for s, d in ei:
        existing.add((min(s, d), max(s, d)))
    
    candidates = []
    np.random.seed(42)
    while len(candidates) < k * 5:
        i = np.random.randint(0, len(node_ids))
        j = np.random.randint(0, len(node_ids))
        if i != j and (min(i,j), max(i,j)) not in existing:
            candidates.append((i, j))
    
    scores = []
    for i, j in candidates:
        s = torch.dot(z[i], z[j]).item()
        scores.append((i, j, torch.sigmoid(torch.tensor(s)).item()))
    
    scores.sort(key=lambda x: -x[2])
    return scores[:k]

new_edges = predict_new_edges(5000)

tsv_path = os.path.join(OUTPUT_DIR, "predicted_edges.tsv")
with open(tsv_path, "w") as f:
    f.write("termA\ttermB\tpredicted_weight\n")
    for i, j, w in new_edges:
        f.write(f"{node_ids[i]}\t{node_ids[j]}\t{w:.6f}\n")
print(f"Wrote {len(new_edges)} predicted edges to {tsv_path}")

In [ ]:
# Export encoder to ONNX
encoder.eval()
encoder_cpu = encoder.cpu()

# ONNX export requires fixed input shapes
dummy_x = torch.randn(data.num_nodes, node_features.shape[1])
dummy_edge_index = data.edge_index.cpu()

onnx_path = os.path.join(OUTPUT_DIR, "model.onnx")
torch.onnx.export(
    encoder_cpu,
    (dummy_x, dummy_edge_index),
    onnx_path,
    input_names=["x", "edge_index"],
    output_names=["embeddings"],
    dynamic_axes={"x": {0: "num_nodes"}, "edge_index": {1: "num_edges"}},
    opset_version=14,
)
print(f"Saved ONNX encoder to {onnx_path}")
print(f"Model size: {os.path.getsize(onnx_path) / 1024:.1f} KB")